In [0]:
from pyspark.sql.functions import when, col

flights_path = "/Volumes/mini_cap/default/airline_data/flights.csv"
bookings_path = "/Volumes/mini_cap/default/airline_data/bookings.csv"
preferences_path = "/Volumes/mini_cap/default/airline_data/Preferences.json"

df_flights = spark.read.option("header", "true").option("inferSchema", "true").csv(flights_path)
df_bookings = spark.read.option("header", "true").option("inferSchema", "true").csv(bookings_path)
df_preferences = (
    spark.read
    .option("multiLine", "true")
    .json(preferences_path)
    .select("passenger_name", "meal", "seat", "extra_baggage")
)

# Transformations
df_bookings_transformed = df_bookings.withColumn(
    "price_band",
    when(col("ticket_price") > 20000, "Premium")
    .when(col("ticket_price") > 10000, "Standard")
    .otherwise("Budget")
)

df_flights_transformed = df_flights.withColumn(
    "delay_flag",
    when(col("status") == "Delayed", "Yes").otherwise("No")
)

# Joins
df_journey = (
    df_bookings_transformed
    .join(df_flights_transformed, on="flight_id", how="inner")
    .join(df_preferences, on="passenger_name", how="left")
)

# Write to Delta table (overwrite for simplicity in this pipeline run)
df_journey.write.format("delta").mode("overwrite").saveAsTable("mini_cap.default.booking_master")

print(f"transform_data complete. Rows written: {df_journey.count()}")

transform_data complete. Rows written: 20
